# Mini-Project: ระบบจองห้องประชุม (Meeting Room Booking System)


## Project Goal:
สร้างโปรแกรมสำหรับจองห้องประชุมแบบ command-line โดยผู้ใช้สามารถ:

    1. ดูรายชื่อห้องประชุมทั้งหมด
    2. ดูตารางการจองของห้องใดห้องหนึ่ง
    3. จองห้องประชุมในช่วงเวลาที่ต้องการ
    4. ยกเลิกการจอง
    5. ออกจากโปรแกรม

## แนวคิดพื้นฐานที่ใช้ (Fundamental Concepts)
- **Tuple (immutable):** รายชื่อห้องประชุมที่มีอยู่จริง เป็นข้อมูล "คงที่" ที่ไม่ควรถูกแก้ไขระหว่างรันโปรแกรม
- **List of Dictionaries (mutable):** เก็บรายการจองแต่ละรายการ เพราะข้อมูลนี้ต้องเพิ่ม/ลบ/แก้ไขได้ตลอดเวลา
- **while loop + if/elif/else:** สร้างเมนูโต้ตอบกับผู้ใช้แบบต่อเนื่อง
- **Defensive programming:** ตรวจสอบก่อนว่าห้อง/เวลาที่จองซ้ำกันหรือไม่ ก่อนอนุญาตให้จอง และตรวจสอบว่ารายการที่จะยกเลิกมีอยู่จริงหรือไม่

## Step 1: Planning (Pseudocode)
1. กำหนดรายชื่อห้องประชุมที่มีอยู่ (tuple, ข้อมูลคงที่)
2. สร้างลิสต์ว่างสำหรับเก็บรายการจอง (list of dict)
3. แสดงเมนูหลักซ้ำๆ ด้วย while loop:
    
    a. จองห้องประชุม

       - รับชื่อห้อง / วันที่ / เวลา / ชื่อผู้จอง
       - ตรวจสอบว่าห้องมีอยู่จริงในรายชื่อห้อง
       - ตรวจสอบว่าห้อง+วันที่+เวลานี้ถูกจองไปแล้วหรือยัง (ป้องกันการจองซ้ำ)
       - ถ้าผ่านการตรวจสอบ ให้เพิ่มรายการจองใหม่ลงในลิสต์
    b. ดูรายการจองทั้งหมด

       - ถ้าลิสต์ว่าง แจ้งว่ายังไม่มีการจอง
       - ถ้ามีข้อมูล ให้วนลูปแสดงผลแต่ละรายการ พร้อมเลขลำดับ
    c. ยกเลิกการจอง

       - แสดงรายการจองทั้งหมดพร้อมเลขลำดับ
       - รับเลขลำดับที่ต้องการยกเลิกจากผู้ใช้
       - ตรวจสอบว่าเลขลำดับที่กรอกถูกต้องหรือไม่ก่อนลบ
    d. ค้นหาการจอง

       - รับคำค้นหา (ชื่อห้อง หรือ วันที่)
       - วนลูปหารายการที่ตรงกับคำค้นหา แล้วแสดงผล
       
    e. ออกจากโปรแกรม
       - หยุด while loop






## Step 2: Code Implementation

In [34]:
ROOMS = ("Room A", "Room B", "Room C", "Room D")
bookings = []

In [35]:
def show_rooms():
    """แสดงรายชื่อห้องประชุมที่มีอยู่ทั้งหมด"""
    print("\nห้องประชุมที่มีอยู่:")
    for room in ROOMS:
        print(f"  - {room}")

In [36]:
def book_room():
    """จองห้องประชุม พร้อมตรวจสอบห้องว่างและการป้อนข้อมูล (Defensive Programming)"""
    print("\n--- 1. จองห้องประชุม ---")
    # 1. รับวัน-เวลาก่อน
    date = input("กรอกวันที่ต้องการจอง (เช่น 2026-08-10): ").strip()
    time = input("กรอกช่วงเวลาที่ต้องการจอง (เช่น 09:00-10:00): ").strip()

    if not date or not time:
        print("❌ กรุณากรอกวันที่และเวลาให้ครบถ้วน")
        return

    # 2. เช็กและแสดงเฉพาะห้องที่ยังว่างอยู่จริง
    available_rooms = get_available_rooms(date, time)

    if not available_rooms:
        print(f"❌ ขออภัย ห้องประชุมเต็มทุกห้องในวันที่ {date} เวลา {time}")
        return

    print(f"\nห้องประชุมที่ยัง 'ว่าง' ในวันที่ {date} เวลา {time}:")
    for r in available_rooms:
        print(f"  - {r}")

    # 3. รับชื่อห้อง และเปรียบเทียบแบบ Case-Insensitive (ไม่สนพิมพ์เล็ก-ใหญ่)
    room_input = input("\nกรอกชื่อห้องที่ต้องการจอง: ").strip()

    matched_room = None
    for r in available_rooms:
        if r.lower() == room_input.lower():
            matched_room = r
            break

    if not matched_room:
        print(f"❌ ไม่อนุญาตให้จอง: ห้อง '{room_input}' ไม่มีในระบบ หรือถูกจองไปแล้วในเวลานี้")
        return

    booked_by = input("กรอกชื่อผู้จอง: ").strip().title()
    if not booked_by:
        print("❌ กรุณากรอกชื่อผู้จอง")
        return

    # 4. บันทึกข้อมูล
    new_booking = {"room": matched_room, "date": date, "time": time, "booked_by": booked_by}
    bookings.append(new_booking)
    print(f"✅ จองห้อง {matched_room} วันที่ {date} เวลา {time} สำเร็จ!")

In [37]:
def view_bookings():
    """แสดงรายการจองทั้งหมดแบบตารางจัดระเบียบ"""
    if not bookings:
        print("\nยังไม่มีรายการจองในระบบ")
        return

    # ปรับ ui
    print("\n========================= รายการจองทั้งหมด =========================")
    print(f"{'ลำดับ':<6} | {'ห้องประชุม':<10} | {'วันที่':<12} | {'เวลา':<13} | {'ผู้จอง':<15}")
    print("-" * 68)
    for i, b in enumerate(bookings, start=1):
        print(f"{i:<6} | {b['room']:<10} | {b['date']:<12} | {b['time']:<13} | {b['booked_by']:<15}")
    print("====================================================================")

In [38]:
def cancel_booking():
    """ยกเลิกการจอง โดยตรวจสอบก่อนว่ารายการที่เลือกมีอยู่จริง"""
    if not bookings:
        print("\nยังไม่มีรายการจองให้ยกเลิก")
        return

    view_bookings()
    choice = input("กรอกลำดับรายการที่ต้องการยกเลิก: ").strip()

    # --- Defensive Programming: ตรวจสอบว่าอินพุตเป็นตัวเลขและอยู่ในช่วงที่ถูกต้อง ---
    if not choice.isdigit():
        print("กรุณากรอกเป็นตัวเลข")
        return

    index = int(choice) - 1
    if index < 0 or index >= len(bookings):
        print("ไม่พบรายการลำดับนี้")
        return

    removed = bookings.pop(index)
    print(f"ยกเลิกการจองห้อง {removed['room']} วันที่ {removed['date']} เวลา {removed['time']} สำเร็จ")

In [39]:
def search_bookings():
    """ค้นหาการจองตามชื่อห้องหรือวันที่"""
    if not bookings:
        print("\nยังไม่มีรายการจองในระบบ")
        return

    keyword = input("กรอกชื่อห้อง หรือ วันที่ ที่ต้องการค้นหา: ").strip().lower()
    results = [
        b for b in bookings
        if keyword in b["room"].lower() or keyword in b["date"].lower()
    ]

    if not results:
        print(f"ไม่พบรายการจองที่ตรงกับ '{keyword}'")
        return

    print(f"\n--- ผลการค้นหา: '{keyword}' ---")
    for i, b in enumerate(results, start=1):
        print(f"{i}. ห้อง: {b['room']} | วันที่: {b['date']} | เวลา: {b['time']} | ผู้จอง: {b['booked_by']}")
    print("-----------------------------")


In [40]:
def get_available_rooms(date, time):

    # ดึงรายชื่อห้องที่ถูกจองแล้วในวัน-เวลานั้น (แปลงเป็นตัวพิมพ์เล็กเพื่อเทียบง่ายขึ้น)
    booked_rooms = {b["room"].lower() for b in bookings if b["date"] == date and b["time"] == time}

    # คืนค่าเฉพาะห้องใน ROOMS ที่ยังไม่ถูกจอง
    return [room for room in ROOMS if room.lower() not in booked_rooms]

In [41]:
def main():
    """เมนูหลักของโปรแกรม (menu-driven interface ด้วย while loop)"""
    print("=== ยินดีต้อนรับสู่ระบบจองห้องประชุม ===")

    while True:
        print("\nเมนูหลัก:")
        print("1. จองห้องประชุม")
        print("2. ดูรายการจองทั้งหมด")
        print("3. ยกเลิกการจอง")
        print("4. ค้นหาการจอง")
        print("5. ออกจากโปรแกรม")

        choice = input("เลือกเมนู (1-5): ").strip()

        if choice == "1":
            book_room()
        elif choice == "2":
            view_bookings()
        elif choice == "3":
            cancel_booking()
        elif choice == "4":
            search_bookings()
        elif choice == "5":
            print("ขอบคุณที่ใช้บริการ ลาก่อน!")
            break
        else:
            print("กรุณาเลือกเมนูที่ถูกต้อง (1-5)")

เรียกใช้โปรแกรม

In [ ]:
main()

=== ยินดีต้อนรับสู่ระบบจองห้องประชุม ===

เมนูหลัก:
1. จองห้องประชุม
2. ดูรายการจองทั้งหมด
3. ยกเลิกการจอง
4. ค้นหาการจอง
5. ออกจากโปรแกรม
เลือกเมนู (1-5): 1

--- 1. จองห้องประชุม ---
กรอกวันที่ต้องการจอง (เช่น 2026-08-10): 2026-08-11
กรอกช่วงเวลาที่ต้องการจอง (เช่น 09:00-10:00): 09:00-10:00

ห้องประชุมที่ยัง 'ว่าง' ในวันที่ 2026-08-11 เวลา 09:00-10:00:
  - Room A
  - Room B
  - Room C
  - Room D

กรอกชื่อห้องที่ต้องการจอง: Room A
กรอกชื่อผู้จอง: parima
✅ จองห้อง Room A วันที่ 2026-08-11 เวลา 09:00-10:00 สำเร็จ!

เมนูหลัก:
1. จองห้องประชุม
2. ดูรายการจองทั้งหมด
3. ยกเลิกการจอง
4. ค้นหาการจอง
5. ออกจากโปรแกรม
เลือกเมนู (1-5): 1

--- 1. จองห้องประชุม ---
กรอกวันที่ต้องการจอง (เช่น 2026-08-10): 2026-08-11
กรอกช่วงเวลาที่ต้องการจอง (เช่น 09:00-10:00): 09:00-10:00

ห้องประชุมที่ยัง 'ว่าง' ในวันที่ 2026-08-11 เวลา 09:00-10:00:
  - Room B
  - Room C
  - Room D

กรอกชื่อห้องที่ต้องการจอง: B
❌ ไม่อนุญาตให้จอง: ห้อง 'B' ไม่มีในระบบ หรือถูกจองไปแล้วในเวลานี้

เมนูหลัก:
1. จองห้องประชุม
2. ดูรายกา

## Step 3: อธิบายแนวคิดที่ใช้ (Explanation of Concepts)

- **`ROOMS = (...)` (tuple):** เก็บรายชื่อห้องประชุมที่มีอยู่จริงในองค์กร ข้อมูลนี้ไม่ควรเปลี่ยนระหว่างการรันโปรแกรม จึงเลือกใช้ tuple (immutable) แทน list (LO1: Data Structure Selection)
- **`bookings = []` (list of dict):** เก็บรายการจองที่เพิ่ม/ลบได้ตลอดเวลา แต่ละรายการเป็น dictionary ที่มีคีย์ `room`, `date`, `time`, `booked_by` ทำให้เข้าถึงข้อมูลแต่ละฟิลด์ได้ง่ายด้วยชื่อคีย์ (LO2: Collection Manipulation)
- **`bookings.append(...)` / `bookings.pop(index)`:** ใช้เพิ่มรายการจองใหม่ และลบรายการจองที่ถูกยกเลิก ตามลำดับ
- **`while True:` ในฟังก์ชัน `main()`:** สร้างลูปเมนูที่ทำงานต่อเนื่องจนกว่าผู้ใช้จะเลือกออกจากโปรแกรม (LO3: Interactive Control Flow)
- **`if/elif/else`:** ใช้เลือกฟังก์ชันที่จะเรียกตามเมนูที่ผู้ใช้เลือก และใช้ตรวจสอบเงื่อนไขต่างๆ เช่น ห้องมีอยู่จริงหรือไม่ อินพุตถูกต้องหรือไม่
- **การตรวจสอบก่อนทำงาน (Defensive Programming):**
  - ใน `book_room()` ตรวจว่าห้องมีอยู่จริง และตรวจว่าห้อง+วันที่+เวลาซ้ำกับรายการที่มีอยู่หรือไม่ ก่อนอนุญาตให้จอง
  - ใน `cancel_booking()` ตรวจว่าอินพุตเป็นตัวเลขและอยู่ในช่วงลำดับที่ถูกต้อง ก่อนลบข้อมูลออกจากลิสต์ (ป้องกัน `IndexError` และ `ValueError`)
  (LO4: Defensive Programming)
- **`search_bookings()`:** ใช้ List Comprehension เพื่อกรองรายการที่ตรงกับคำค้นหา แสดงให้เห็นการประยุกต์ใช้ list + string methods ร่วมกัน